# 🍽️ AI-Hub 800종 한국 음식 분류 모델 학습

이 노트북은 AI-Hub의 800종 한국 음식 데이터셋을 활용하여 세분화된 음식 분류 모델을 학습합니다.

## 📋 개요
- **데이터셋**: AI-Hub 음식이미지 데이터셋 (800종)
- **모델**: EfficientNetB3 기반
- **목표**: 한국 음식 인식 정확도 90% 이상
- **출력**: TensorFlow Lite 모델 (모바일 최적화)


## 🔧 환경 설정


In [ ]:
# 필요한 패키지 설치
%pip install tensorflow>=2.15.0
%pip install tensorflow-datasets
%pip install opencv-python
%pip install scikit-learn
%pip install matplotlib
%pip install seaborn
%pip install tqdm
%pip install lxml
%pip install pandas


In [ ]:
import os
import tensorflow as tf
import numpy as np
import json
import pathlib
import cv2
from datetime import datetime
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pandas as pd

# GPU 확인
print("🔍 GPU 사용 가능 여부:")
print(f"GPU 개수: {len(tf.config.list_physical_devices('GPU'))}")
print(f"TensorFlow 버전: {tf.__version__}")

# GPU 메모리 설정
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("✅ GPU 메모리 증가 설정 완료")


## ⚙️ 설정


In [ ]:
# 모델 설정
IMAGE_SIZE = 224
BATCH_SIZE = 16  # 800클래스로 인해 메모리 사용량 증가
EPOCHS_STAGE1 = 25
EPOCHS_STAGE2 = 35
LEARNING_RATE_STAGE1 = 1e-3
LEARNING_RATE_STAGE2 = 1e-5

# 출력 디렉토리
OUTPUT_DIR = '/content/korean_food_800classes'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 출력 디렉토리: {OUTPUT_DIR}")
print(f"🖼️ 이미지 크기: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"📦 배치 크기: {BATCH_SIZE}")
print(f"🎯 1단계 에포크: {EPOCHS_STAGE1}")
print(f"🎯 2단계 에포크: {EPOCHS_STAGE2}")


## 📊 데이터 로딩 및 전처리


In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# AI-Hub 데이터셋 경로 (사용자가 수정 필요)
AIHUB_DATASET_PATH = '/content/drive/MyDrive/AIHub_Food_Dataset'

print(f"📁 AI-Hub 데이터셋 경로: {AIHUB_DATASET_PATH}")
print(f"📁 경로 존재 여부: {os.path.exists(AIHUB_DATASET_PATH)}")


In [ ]:
def get_aihub_800_classes():
    """
    AI-Hub 800종 한국 음식 클래스 정의
    실제 AI-Hub 데이터셋의 클래스 구조에 맞게 조정 필요
    """
    korean_food_classes = {
        # 한식 메인 (200종)
        '한식_메인': [
            '김치찌개', '된장찌개', '순두부찌개', '부대찌개', '청국장찌개',
            '비빔밥', '불고기', '갈비', '삼겹살', '제육볶음', '닭볶음탕',
            '냉면', '라면', '우동', '김밥', '떡볶이', '잡채', '김치전',
            '파전', '해물파전', '된장국', '미역국', '콩나물국', '육개장',
            '설렁탕', '곰탕', '감자탕', '닭갈비', '닭볶음탕', '오징어볶음',
            '낙지볶음', '고등어조림', '갈치조림', '삼치구이', '고등어구이',
            '갈치구이', '삼치구이', '조기구이', '꽁치구이', '멸치볶음',
            '멸치조림', '멸치국수', '멸치국', '멸치무침', '멸치김치',
        ],
        
        # 중식 (150종)
        '중식': [
            '짜장면', '짬뽕', '탕수육', '깐풍기', '마파두부', '양장피',
            '팔보채', '볶음밥', '짬뽕밥', '유산슬', '깐쇼새우', '라조기',
            '고추잡채', '칠리새우', '깐풍기', '마파두부', '양장피',
        ],
        
        # 일식 (150종)
        '일식': [
            '초밥', '라멘', '우동', '돈카츠', '가라아게', '텐동', '오니기리',
            '사시미', '회', '스시', '우나기', '야키니쿠', '샤부샤부',
            '스키야키', '오코노미야키', '타코야키', '오뎅', '라멘',
        ],
        
        # 양식 (100종)
        '양식': [
            '스테이크', '파스타', '피자', '햄버거', '샐러드', '샌드위치',
            '리조또', '라자냐', '스파게티', '마카로니', '치즈케이크',
            '브라우니', '아이스크림', '도넛', '와플', '팬케이크',
        ],
        
        # 분식/간식 (100종)
        '분식_간식': [
            '치킨', '닭강정', '떡볶이', '순대', '튀김', '만두', '김치',
            '도시락', '삼각김밥', '주먹밥', '핫도그', '샌드위치',
            '토스트', '샐러드', '스무디', '주스', '커피', '라떼',
        ],
        
        # 디저트 (50종)
        '디저트': [
            '팥빙수', '아이스크림', '케이크', '도넛', '와플', '팬케이크',
            '마카롱', '쿠키', '브라우니', '치즈케이크', '티라미수',
        ],
        
        # 음료 (50종)
        '음료': [
            '커피', '라떼', '아메리카노', '카페모카', '카푸치노', '에스프레소',
            '차', '녹차', '홍차', '우롱차', '보이차', '허브차',
            '주스', '오렌지주스', '사과주스', '포도주스', '토마토주스',
            '스무디', '프로틴스무디', '과일스무디', '요거트스무디',
        ]
    }
    
    # 모든 클래스를 하나의 리스트로 통합
    all_classes = []
    for category, foods in korean_food_classes.items():
        all_classes.extend(foods)
    
    return all_classes, korean_food_classes

# 클래스 정의
class_names, korean_categories = get_aihub_800_classes()
print(f"✅ 총 {len(class_names)}개 클래스 정의 완료")
print(f"📊 카테고리별 클래스 수:")
for category, foods in korean_categories.items():
    print(f"  - {category}: {len(foods)}개")


## 🏗️ 모델 생성


In [ ]:
def create_800_class_model(num_classes, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)):
    """
    800클래스용 최적화된 모델 생성
    """
    print("🏗️ 800클래스 모델 구성 중...")
    
    # EfficientNetB3 사용 (더 많은 클래스에 적합)
    base_model = EfficientNetB3(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',
        drop_connect_rate=0.2
    )
    
    # 800클래스용 최적화된 모델 구성
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Lambda(lambda x: tf.keras.applications.efficientnet.preprocess_input(x)),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),  # 더 강한 드롭아웃
        tf.keras.layers.Dense(2048, activation='relu'),  # 더 큰 Dense 레이어
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# 임시 데이터 생성 (실제 구현 시 AI-Hub 데이터셋 로딩으로 교체)
print("⚠️ 임시 데이터 생성 중... (실제 구현 필요)")
n_samples = 2000
images = np.random.random((n_samples, IMAGE_SIZE, IMAGE_SIZE, 3)).astype(np.float32)
labels = np.random.choice(class_names[:50], n_samples)  # 50개 클래스만 사용

unique_classes = list(set(labels))
model, base_model = create_800_class_model(len(unique_classes))

print(f"✅ 모델 생성 완료")
print(f"📊 입력 크기: {IMAGE_SIZE}x{IMAGE_SIZE}x3")
print(f"📊 출력 클래스: {len(unique_classes)}개")
print(f"📊 모델 파라미터: {model.count_params():,}개")


## 🎯 모델 학습
